# Math targets — quick sampling demo

Sample from `make_gaussian`, `make_banana`, and `make_gaussian_mixture` with all four PDMPs and overlay the known marginals.

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

if Path.cwd().name == "notebooks":
    os.chdir("..")

from sazz.models import make_gaussian, make_banana, make_gaussian_mixture, make_neals_funnel
from sazz.utils.glm_utils import build_sampler, resample, set_seed
from sazz.utils.sampling import (
    resample_boomerang_path, #resample_boomerang_path_sticky,
    resample_zigzag_path, #resample_zigzag_path_sticky,
)

torch.set_default_dtype(torch.float64)

PDMP_SPECS = [
    ("Boomerang",     "boomerang", False, "C0"),
    #("Sticky-Boom",   "boomerang", True,  "C1"),
    ("ZigZag",        "zigzag",    False, "C9"),
    #("Sticky-ZZ",     "zigzag",    True,  "C3"),
]

In [ ]:
from dataclasses import dataclass

@dataclass
class Cfg:
    n_skel:      int   = 10_000
    n_resample:  int   = 5_000
    burnin_frac: float = 0.2
    refresh_rate: float = 1.0
    kappa_null:  float = 0.4
    kappa_int:   float = 1e6   # unused for math targets (no intercept)
    thinning:    str   = "brent_monotone"
    t_max_zz:    float = 0.5
    gamma_zz:    float = 0.01
    seed:        int   = 0

cfg = Cfg()


def sample_all(target, cfg):
    """Run all four PDMPs on target; return dict name -> samples array [N, D]."""
    kappa = torch.full((target.D,), cfg.kappa_null, dtype=torch.float64)
    x_ref_np = target.x_ref.cpu().numpy()
    results = {}
    for name, family, sticky, color in PDMP_SPECS:
        set_seed(cfg.seed)
        s = build_sampler(family, sticky, target, cfg, kappa)
        if family == "boomerang":
            res = s.sample(N=cfg.n_skel, x0=target.x_ref.clone(), diagnostics=False)
            #res = s.sample(N=cfg.n_skel*2, x0=target.x_ref.clone(), diagnostics=False)
        else:
            res = s.sample(N=cfg.n_skel, x0=target.x_ref.clone(), diagnostics=False)
        samples = resample(family, sticky, res, x_ref_np, cfg)
        results[name] = {"samples": samples, "color": color, "sticky": sticky}
        #print(f"  {name:<14} n_bounces={res['n_bounces']}  draws={samples.shape[0]}")
    return results

In [ ]:
def plot_marginals(target, results, max_coords=4, bins=60):
    """Histogram of each coordinate vs the analytic marginal PDF."""
    coords = list(target.marginal_grids.keys())[:max_coords]
    n_samplers = len(results)
    fig, axes = plt.subplots(
        len(coords), n_samplers,
        figsize=(3.2 * n_samplers, 2.4 * len(coords)),
        squeeze=False,
    )
    for row, i in enumerate(coords):
        mg = target.marginal_grids[i]
        for col, (name, r) in enumerate(results.items()):
            ax = axes[row, col]
            ax.hist(r["samples"][:, i], bins=bins, density=True,
                    color=r["color"], alpha=0.55, label=name)
            ax.plot(mg["grid"], mg["pdf"], "k-", lw=1.5, label="true")
            ax.set_xlabel(mg["label"], fontsize=9)
            ax.set_ylabel("density" if col == 0 else "", fontsize=8)
            if row == 0:
                ax.set_title(name, fontsize=9)
            for spine in ("top", "right"):
                ax.spines[spine].set_visible(False)
    fig.suptitle(target.name, fontsize=11, y=1.01)
    fig.tight_layout()
    plt.show()

## 0. Gaussian (diagonal)

The reference measure equals the target, so the residual gradient is identically zero. The Boomerang should report **zero bounces**.

In [ ]:
target_gauss = make_gaussian(D=5, cov="diagonal")
print(f"Target: {target_gauss.name}  D={target_gauss.D}")
results_gauss = sample_all(target_gauss, cfg)

In [ ]:
plot_marginals(target_gauss, results_gauss, max_coords=3)

## 1. Gaussian (non-diagonal)
Could be nice to verify the reference measure is good for the Boomerang

In [ ]:
target_gauss_dense = make_gaussian(D=5, cov="random")
print(f"Target: {target_gauss_dense.name}  D={target_gauss_dense.D}")
print(f"Sigma_inv is dense: {target_gauss_dense.Sigma_inv.ndim == 2}")
results_gauss_dense = sample_all(target_gauss_dense, cfg)


In [ ]:
plot_marginals(target_gauss_dense, results_gauss_dense, max_coords=3)

## 2. Banana (Rosenbrock)

Non-Gaussian, curved geometry. Good stress test for the sampler's ability to follow a non-linear manifold.

In [ ]:
target_banana = make_banana(a=1.0, scale=2.0)
print(f"Target: {target_banana.name}  D={target_banana.D}")
results_banana = sample_all(target_banana, cfg)

In [ ]:
plot_marginals(target_banana, results_banana)

# Also show the 2-D joint scatter for the banana
fig, axes = plt.subplots(1, len(results_banana), figsize=(3.2 * len(results_banana), 3), squeeze=False)
for col, (name, r) in enumerate(results_banana.items()):
    ax = axes[0, col]
    ax.scatter(r["samples"][:, 0], r["samples"][:, 1],
               s=2, alpha=0.3, color=r["color"], rasterized=True)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel(r"$\beta_1$"); ax.set_ylabel(r"$\beta_2$" if col == 0 else "")
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
fig.suptitle(f"{target_banana.name} — joint scatter", fontsize=10, y=1.01)
fig.tight_layout()
plt.show()

## 3. Gaussian mixture (bimodal)

Two well-separated modes. Tests whether samplers can cross the low-density valley.

In [ ]:
target_mix = make_gaussian_mixture(D=1, preset="bimodal")
print(f"Target: {target_mix.name}  D={target_mix.D}")
results_mix = sample_all(target_mix, cfg)

In [ ]:
plot_marginals(target_mix, results_mix)

## 4. Neals funnel

In [ ]:
target_funnel = make_neals_funnel(D=2)
print(f"Target: {target_funnel.name}  D={target_funnel.D}")
results_funnel = sample_all(target_funnel, cfg)

In [ ]:
plot_marginals(target_funnel, results_funnel)